# Triple Scoring Function - DisMult & MRR

In [ ]:
# DONE Email Xander to move meeting
# DONE Double check code 
    # DONE check edge_index --> correct index for transactions?
    # DONE Check embeddings 
    # DONE Manipulate Timestamp feature for edges 
# DONE DisMult 
# DONE Ranking via MRR

# DONE register account for DAS5

# DONE check theoretical concepts
    # DONE DisMult, MRR
    # DONE mini batches, optimization and loss calculations 
# DONE Review Code --> Diminishing Training effectiveness --> architecture correct? --> multiplication of embeddings & score between 0 and 1 CORRECT? 
# DONE improvements 
    # DONE normalize edge features (timestamp) and currency
    # TODO mini batches 

# DONE new system 
    # DONE Simplify GNN to 1-2 convolutional layer (prevent overfitting)
    # DONE Take embeddings directly from the layer --> in forward() implement changes --> Dismult function that takes embeddings
    # DONE DisMult function (simple scoring function)
    # DONE Put in LIST --> loss function compare predictions and labels 
    # DONE Evaluation function using BCELogitcs 
        # TODO negative sampling --> comparing TRUE with FALSE -->  
    # DONE MRR for evaluating --> every 5th epoch 



# TODO Send the Code to Supervisor for a check 
    # TODO Say what has been done 
    # TODO The scores may not be updating correctly, where is the problem? 



# TODO MRR fix 
# TODO Time should be enhanced in comparison to other features --> more weight? taken more into account 
# TODO related work / literature review --> don't focus too much on graph learning, focus more on link 
# predicitons with weights (features) (weighted learning) --> short introduction on ML in money-laundering (keep it short) 
# --> graph-based methods for money-laundering 

# TODO Dismult --> knowledge graph embedding method --> translational models:
    # TODO cannot caption the direction of the arrow --> 
# TODO TransE --> ComplEx --> HoLE 

# TODO keep the comparison narrow --> don't do a broad comparisons
    # TODO we can say that we have looked into other models/structures but we are trying to... (achieve goal in a specific way)
    # TODO we can also point to work done by other research

 

In [ ]:
import pickle
import gzip
import torch
import torch.nn as nn
import torch.optim as optim

![Screenshot 2024-05-23 at 21.28.02.png](<attachment:Screenshot 2024-05-23 at 21.28.02.png>)

## Load data

In [ ]:
with open("graph.pickle", "rb") as f:
    loaded_data = pickle.load(f)

labels = loaded_data["labels"]

In [ ]:
def print_rdf_triples(file_path):
    with gzip.open(file_path, 'rt') as f:
        for line in f:
            print(line.strip())

# Replace 'embeddings.nt.gz' with the actual file path
print_rdf_triples("embeddings.nt.gz")

In [ ]:
def process_rdf(file_path):
    edges = []
    embeddings = []
    node_set = set()
    relation_set = set()

    with gzip.open(file_path, 'rt') as f:
        new_list = []
        counter = 0
        for line in f:
            if 'connectedTo' in line:  # edge
                edges.append(line.strip())
                parts = line.strip().split()
                head = parts[0].strip('<>').split('/')[-1]
                tail = parts[2].strip('<>').split('/')[-1]

                node_set.add(head)
                node_set.add(tail)
            elif 'hasEmbedding' in line:  # embeddings
                node_embedding = [float(val) for val in line.split('"')[1].split()]
                # node_embedding = torch.tensor(node_embedding)
                new_list.append(node_embedding)
                counter += 1
                if counter == 3:
                    embeddings.append(new_list)
                    new_list = []
                    counter = 0
    return edges, embeddings, node_set

# Test the function
edges, embeddings, node_set = process_rdf("embeddings.nt.gz")

# Prepare data for training
heads = []
relations = []
tails = []

for i in range(len(embeddings)):
    head = embeddings[i][0]
    tail = embeddings[i][1]
    relation = embeddings[i][2]
    
    heads.append(head)
    tails.append(tail)
    relations.append(relation)

# Stack the tensors for batch processing
heads = torch.tensor(heads, dtype=torch.float32, requires_grad=True)
relations = torch.tensor(relations, dtype=torch.float32, requires_grad=True)
tails = torch.tensor(tails, dtype=torch.float32, requires_grad=True)

print(heads.size())
print(tails.size())
print(relations.size())

# Get the number of nodes and relations from the data
num_nodes = len(node_set)
num_relations = len(embeddings)
embedding_dim = heads[0].size(0)  # Dimensionality of embeddings

print(num_nodes)
print(num_relations)
print(embedding_dim)


In [ ]:
class Dismult(nn.Module):
    def __init__(self, embedding_size):
        super(Dismult, self).__init__()
        self.embedding_size = embedding_size
        self.linear = nn.Linear(embedding_size * 3, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, head, relation, tail):
        combined = torch.cat((head, relation, tail), dim=1)
        out = self.linear(combined)
        score = self.sigmoid(out)
        return score

def calculate_mrr(ranks):
    reciprocal_ranks = 1.0 / ranks
    return torch.mean(reciprocal_ranks)

# Instantiate model
model = Dismult(embedding_size=15)

# Define loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training loop
num_epochs = 10000
for epoch in range(num_epochs):
    optimizer.zero_grad()
    scores = model(heads, relations, tails)

    # print(scores.size())
    # print(scores.squeeze().size())

    loss = criterion(scores.squeeze(), labels)
    loss.backward()
    optimizer.step()

    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item()}')

# After training, you can use the model to make predictions


In [ ]:
with torch.no_grad():
    scores = model(heads, relations, tails).squeeze()
    sorted_indices = torch.argsort(scores, descending=True)
    true_indices = sorted_indices[:torch.sum(labels).int()]  # Select top predictions based on number of positive labels
    mrr = 0
    for i, idx in enumerate(true_indices):
        if labels[idx].item() == 1:
            mrr = 1 / (i + 1)  # MRR calculation
            break
    print("Mean Reciprocal Rank (MRR):", mrr)


In [ ]:
print("Sorted Indices:", sorted_indices)
print("True Indices:", true_indices)

In [ ]:
with torch.no_grad():
    scores = model(heads, relations, tails).squeeze()
    sorted_indices = torch.argsort(scores, descending=True)
    true_indices = sorted_indices[:50]  # Select top 5 predictions
    mrr = 0
    for i, idx in enumerate(true_indices):
        if labels[idx].item() == 1:
            mrr = 1 / (i + 1)  # MRR calculation
            break
    print("Mean Reciprocal Rank (MRR):", mrr)


In [ ]:
true_indices

In [ ]:
torch.nonzero(labels == 1).squeeze()